# Gold Layer - Business Metrics & Aggregations

1. **Technician Performance**: Highlighting workloads, average repair durations, and delay rates.
2. **Delay Analysis**: Visualizing how often promised dates are missed and by how many days.
3. **Repeat Customers**: Finding customers returning multiple times for the same issue.
4. **Device Brand Analysis**: Analyzing repair volumes and trends across device categories and brands.
5. **Customer Latest Visit**: Spotting the most recent job status and details for each customer.

### Load Shared Configuration and Helper Functions
Loads config values (table names, thresholds) and the `save_as_delta` / `logger` helpers from `00_Config_Utils`.

In [0]:
%run ./00_Config_Utils

# logging - Shared Configuration and Helper Functions

This keeps the project **modular** (reusable functions instead of copy-pasted code) and **parameterized** (thresholds and table names live in one place, not hardcoded everywhere).

We use Python's built-in `logging` module instead of plain `print()` statements. This gives every message a timestamp and a severity level (INFO, WARNING, ERROR), which is standard practice in real data pipelines.

Every layer (Bronze, Silver, Gold) saves DataFrames as Delta tables the same way. This function avoids repeating that logic in every notebook.

### Step 1: Set Up Logging


2026-07-11 19:10:27,237 - INFO - Logger initialized for ServiceTrack pipeline.


### Step 2: Configuration Values (Parameterized Settings)


2026-07-11 19:10:27,655 - INFO - Configuration values loaded.


### Step 3: Reusable Function - Load a CSV File


### Step 4: Reusable Function - Save a DataFrame as a Delta Table


### Step 5: Reusable Function - Convert Multiple Columns to Date Type


### Step 6: Reusable Function - Validate Row Counts


### Import Window and Aggregation Functions


In [0]:
from pyspark.sql.functions import col, count, avg, sum, when, row_number, datediff
from pyspark.sql.window import Window

### Read Silver Enriched Data

In [0]:
# Read Silver dataset using config table name, wrapped in error handling
try:
    df_silver = spark.read.table(SILVER_ENRICHED_JOBS_TABLE)
    logger.info(f"Silver dataset loaded: {df_silver.count()} rows")
except Exception as e:
    logger.error(f"Failed to read Silver table: {e}")
    raise

2026-07-11 19:10:30,724 - INFO - Silver dataset loaded: 1500 rows


### Metric 1 - Technician Performance

In [0]:
# Compute technician performance statistics
df_tech_performance = df_silver.groupBy("technician_id", "technician_name") \
    .agg(
        count("job_id").alias("total_jobs"),
        sum(when(col("job_status") == "Completed", 1).otherwise(0)).alias("completed_jobs"),
        avg("Repair_Duration").alias("avg_repair_duration_days"),
        sum(when(col("completed_date") > col("promised_date"), 1).otherwise(0)).alias("delayed_jobs")
    ) \
    .withColumn(
        "delay_rate_pct",
#avoid divide-by-zero when a technician has 0 completed jobs (returns null instead)
        when(col("completed_jobs") > 0, (col("delayed_jobs") / col("completed_jobs")) * 100).otherwise(None)
    )

# Display the metrics
display(df_tech_performance.limit(5))

technician_id,technician_name,total_jobs,completed_jobs,avg_repair_duration_days,delayed_jobs,delay_rate_pct
T005,Kavitha Nair,192,143,2.569620253164557,0,0.0
T006,Deepak Sharma,186,135,5.246478873239437,63,46.666666666666664
T007,Meena Reddy,209,152,4.166666666666667,24,15.789473684210526
T008,Arjun Iyer,159,111,7.094827586206897,88,79.27927927927928
T004,Amit Patel,164,119,6.4609375,90,75.63025210084034


### Metric 2 - Delay Analysis

In [0]:
#calculate actual delay using promised_date directly, instead of a hardcoded '-5' SLA assumption
df_delays = df_silver.withColumn(
    "days_delayed",
    when(col("completed_date") > col("promised_date"), datediff(col("completed_date"), col("promised_date"))).otherwise(0)
)

# Metric A: average delay across ALL jobs (includes on-time jobs counted as 0)
# Metric B: average delay ONLY among jobs that were actually delayed (more meaningful for SLA review)
df_delay_analysis = df_delays.agg(
    avg("days_delayed").alias("avg_delay_days_all_jobs"),
    avg(when(col("days_delayed") > 0, col("days_delayed"))).alias("avg_delay_days_delayed_only"),
    (sum(when(col("completed_date") > col("promised_date"), 1).otherwise(0)) / count("job_id") * 100).alias("overall_delay_rate_pct")
)

display(df_delay_analysis)

2026-07-11 19:10:44,090 - INFO - Received command c on object id p0


avg_delay_days_all_jobs,avg_delay_days_delayed_only,overall_delay_rate_pct
0.44333333333333336,2.239057239057239,19.8


###  Metric 3 - Repeat Customers

In [0]:
# Count jobs gruped by customer and issue category
df_repeat_customers = df_silver.groupBy("customer_id", "customer_name", "issue_type") \
    .agg(count("job_id").alias("visit_count")) \
    .filter(col("visit_count") > 1) \
    .sort(col("visit_count").desc())

display(df_repeat_customers.limit(5))

customer_id,customer_name,issue_type,visit_count
CUST0231,Neha Fernandez,Charging Port Fault,6
CUST0035,Gaurav Chaudhary,Wi-Fi Not Connecting,5
CUST0274,Vimala Qureshi,RAM Issue,4
CUST0045,Zara Mishra,Charging Port Fault,4
CUST0192,Sonam Iyer,Wi-Fi Not Connecting,4


### Metric 3b - Repeat Customers (Overall, All Issue Types Combined)
The metric above only counts repeat visits for the *same* issue type. This additional metric counts a customer as a repeat customer if they visited **for any reason more than once** — a broader view of customer retention.

In [0]:
# Count total jobs per customer, regardless of issue_type
df_repeat_customers_overall = df_silver.groupBy("customer_id", "customer_name") \
    .agg(count("job_id").alias("total_visit_count")) \
    .filter(col("total_visit_count") > 1) \
    .sort(col("total_visit_count").desc())

display(df_repeat_customers_overall.limit(5))

2026-07-11 19:10:47,951 - INFO - Received command c on object id p0


customer_id,customer_name,total_visit_count
CUST0291,Rohan Bhat,14
CUST0052,Ashok Bhat,14
CUST0231,Neha Fernandez,14
CUST0300,Chandana Shetty,14
CUST0146,Prakash Shetty,14


### Metric 4 - Device Brand Analysis.

In [0]:
# Aggregate volume of repair jobs by brand and type
df_brand_analysis = df_silver.groupBy("brand", "device_type", "issue_type") \
    .agg(
        count("job_id").alias("total_jobs"),
        avg("estimated_cost").alias("avg_estimated_cost"),
        avg("actual_cost").alias("avg_actual_cost")
    ) \
    .sort(col("total_jobs").desc())

display(df_brand_analysis.limit(5))

brand,device_type,issue_type,total_jobs,avg_estimated_cost,avg_actual_cost
Motorola,Printer,Wi-Fi Not Connecting,9,3383.561111111111,5054.2675
Sony,Monitor,Charging Port Fault,9,5131.721111111112,4103.24875
Lenovo,Monitor,Wi-Fi Not Connecting,8,3906.43625,5122.405000000001
Lenovo,Microwave,Water Damage,7,3335.871428571428,3579.2766666666666
HP,Monitor,Overheating,6,4205.236666666667,5772.4775


### Metric 5 - Customer Latest Visit

In [0]:
# Create window partition to rank visits per customer
window_spec = Window.partitionBy("customer_id").orderBy(col("received_date").desc(), col("job_id").desc())

# Apply window row_number ranking
df_customer_latest_visit = df_silver.withColumn("row_num", row_number().over(window_spec)) \
    .filter(col("row_num") == 1) \
    .select("customer_id", "customer_name", "job_id", "received_date", "job_status", "actual_cost")

display(df_customer_latest_visit.limit(5))

customer_id,customer_name,job_id,received_date,job_status,actual_cost
CUST0001,Kiran Patel,JOB00991,2024-03-13,Cancelled,null
CUST0003,Bhavna Tiwari,JOB00107,2024-03-18,Completed,2193.57
CUST0004,Syed Rao,JOB00193,2024-03-16,Pending,null
CUST0005,Suresh Bhat,JOB00553,2024-02-20,Completed,6572.34
CUST0006,Deepika Iyer,JOB00036,2024-03-10,Cancelled,null


### Save Business Metrics as Gold Delta Tables

In [0]:
# Save all Gold tables using the shared save_as_delta() helper and config table names,wrapped in error handling so a failure on one table is logged clearly.
try:
    save_as_delta(df_tech_performance, GOLD_TECHNICIAN_PERFORMANCE_TABLE)
    save_as_delta(df_delay_analysis, GOLD_DELAY_ANALYSIS_TABLE)
    save_as_delta(df_repeat_customers, GOLD_REPEAT_CUSTOMERS_TABLE)
    save_as_delta(df_repeat_customers_overall, GOLD_REPEAT_CUSTOMERS_OVERALL_TABLE)
    save_as_delta(df_brand_analysis, GOLD_DEVICE_BRAND_ANALYSIS_TABLE)
    save_as_delta(df_customer_latest_visit, GOLD_CUSTOMER_LATEST_VISIT_TABLE)
    logger.info("All 6 Gold Delta Tables saved successfully.")
except Exception as e:
    logger.error(f"Failed while saving Gold tables: {e}")
    raise

2026-07-11 19:10:56,989 - INFO - Saved Delta table: gold_technician_performance
2026-07-11 19:10:59,982 - INFO - Saved Delta table: gold_delay_analysis
2026-07-11 19:11:03,671 - INFO - Saved Delta table: gold_repeat_customers
2026-07-11 19:11:09,513 - INFO - Saved Delta table: gold_repeat_customers_overall
2026-07-11 19:11:12,552 - INFO - Saved Delta table: gold_device_brand_analysis
2026-07-11 19:11:15,662 - INFO - Saved Delta table: gold_customer_latest_visit
2026-07-11 19:11:15,662 - INFO - All 6 Gold Delta Tables saved successfully.
